# Limpeza de `assinaturas.csv`

**Objetivo:** gerar a tabela de assinaturas pronta para análise em `dados/processados/assinaturas_limpa.parquet`.

**Regras aplicadas** (todas registradas no `decisoes.md`):

| Passo | Regra | Origem |
|---|---|---|
| 1 | Tirar as contas de teste (e-mail `@vitaliza.com.br`) | decisão de 31/10/2026 |
| 2 | Converter `inicio` e `data_fim` para data; `motivo_fim` vazio vira ausente | tipos |
| 3 | Criar `ativa_no_corte`, `duracao_dias` e `duracao_censurada` | `status_atual` e `renovacao_automatica` do retrato valem para o corte |
| 4 | Marcar `inicio_antes_do_log` (início antes de 01/07/2024) | coorte incompleta antes do log |

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.
As linhas antes e depois de cada passo vão para `saidas/limpeza_assinaturas_registro.csv`.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
INICIO_LOG = pd.Timestamp("2024-07-01")   # primeiro dia de eventos_assinatura.csv
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)  # 01/10/2026 00:00, horário de Brasília
TZ_BR = "America/Sao_Paulo"

ARQUIVOS = ["assinaturas.csv", "usuarios.csv", "eventos_assinatura.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas)")
    return df

## 0. Leitura

Tudo como texto (`dtype=str, keep_default_na=False`): o arquivo chega como foi gravado e vazio fica `""`.

In [2]:
assinaturas = pd.read_csv(BRUTOS / "assinaturas.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False)

assert len(assinaturas) == 32_643   # cabeçalho da seção 03 do dicionário
assert len(usuarios) == 64_948
assert assinaturas["assinatura_id"].is_unique
assert assinaturas["user_id"].isin(usuarios["user_id"]).all()

base = registrar("0. Arquivo bruto", "assinaturas.csv sem filtro", assinaturas)

0. Arquivo bruto: 32,643 linhas (0 removidas)


## 1. Contas de teste

As 60 assinaturas de plano `cortesia`, canal `interno`, são das contas de teste da própria Vitaliza.
Elas também são as 42 assinaturas com início anterior ao cadastro da conta: saem juntas.

In [3]:
email = usuarios.set_index("user_id")["email"].str.strip().str.lower()
eh_teste = base["user_id"].map(email).str.endswith("@vitaliza.com.br")

cadastro = pd.to_datetime(base["user_id"].map(usuarios.set_index("user_id")["data_cadastro"]),
                          format="%Y-%m-%d %H:%M:%S")
inicio_antes_cadastro = pd.to_datetime(base["inicio"], format="%Y-%m-%d") < cadastro.dt.normalize()

display(pd.crosstab([base["plano_atual"], base["canal_cobranca"]], eh_teste.rename("conta de teste")))
print(f"início antes do cadastro: {inicio_antes_cadastro.sum()} linhas, "
      f"{(inicio_antes_cadastro & eh_teste).sum()} delas de conta de teste")

assert eh_teste.sum() == 60
assert (eh_teste == (base["plano_atual"] == "cortesia")).all()
assert (inicio_antes_cadastro <= eh_teste).all()

base = registrar("1. Sem contas de teste", "e-mail não termina em @vitaliza.com.br", base.loc[~eh_teste].copy())

conta de teste              False  True 
plano_atual canal_cobranca              
anual       app_store        4552      0
            google_play      4529      0
            web_cartao       2859      0
            web_pix          1087      0
cortesia    interno             0     60
mensal      app_store        6618      0
            google_play      6419      0
            web_cartao       4228      0
            web_pix          2291      0

início antes do cadastro: 42 linhas, 42 delas de conta de teste
1. Sem contas de teste: 32,583 linhas (60 removidas)


## 2. Tipos

As categorias já chegam padronizadas (minúsculas, sem espaços): a célula confere e não altera nada.
Datas em `AAAA-MM-DD`; `motivo_fim` vazio (assinatura ativa) vira ausente.

In [4]:
CATEGORICAS = {
    "plano_atual": {"mensal", "anual"},
    "canal_cobranca": {"app_store", "google_play", "web_cartao", "web_pix"},
    "status_atual": {"ativa", "expirada", "reembolsada"},
    "renovacao_automatica": {"sim", "nao"},
    "motivo_fim": {"", "renovacao_desligada", "falha_pagamento", "chargeback", "reembolso"},
}
for col, permitidos in CATEGORICAS.items():
    assert (base[col] == base[col].str.strip().str.lower()).all(), col
    assert set(base[col]) <= permitidos, (col, set(base[col]) - permitidos)

base["inicio"] = pd.to_datetime(base["inicio"], format="%Y-%m-%d")
base["data_fim"] = pd.to_datetime(base["data_fim"].replace("", None), format="%Y-%m-%d")
base["motivo_fim"] = base["motivo_fim"].replace("", None)

assert base["inicio"].notna().all()
print(f"inicio: {base['inicio'].min().date()} a {base['inicio'].max().date()}")
print(f"data_fim: {base['data_fim'].notna().sum():,} preenchidas, "
      f"{base['data_fim'].min().date()} a {base['data_fim'].max().date()}")
assert base["inicio"].max() <= CORTE and base["data_fim"].max() <= CORTE

base = registrar("2. Tipos convertidos", "inicio e data_fim como data; motivo_fim vazio vira ausente", base)

inicio: 2020-09-08 a 2026-09-30


data_fim: 17,830 preenchidas, 2024-07-07 a 2026-09-30
2. Tipos convertidos: 32,583 linhas (0 removidas)


## 3. Situação no corte e duração

Nenhuma assinatura começa ou termina depois de 30/09/2026, e o log não tem evento depois do corte
(nem `renovacao_automatica_desligada` em 01/10/2026). Então `status_atual` e `renovacao_automatica`
do retrato de 01/10/2026 descrevem a assinatura no fim do dia do corte.

- `ativa_no_corte`: `status_atual == "ativa"` (exatamente as que não têm `data_fim`).
- `duracao_dias`: de `inicio` até `data_fim`; para as ativas, até o corte.
- `duracao_censurada`: `True` nas ativas, cuja duração real ainda não acabou (não entram como "duração até sair").

In [5]:
def para_brasilia(s):
    """Texto de data e hora para horário de Brasília: com 'Z' é UTC, sem 'Z' já está em Brasília."""
    utc = s.str.endswith("Z")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    out[utc] = pd.to_datetime(s[utc], format="%Y-%m-%dT%H:%M:%SZ", utc=True).dt.tz_convert(TZ_BR).dt.tz_localize(None)
    out[~utc] = pd.to_datetime(s[~utc], format="%Y-%m-%d %H:%M:%S")
    return out

eventos = pd.read_csv(BRUTOS / "eventos_assinatura.csv", dtype=str, keep_default_na=False)
eventos["data_br"] = para_brasilia(eventos["data_hora"])
assert eventos["data_br"].notna().all()
print(f"eventos: {len(eventos):,} linhas, de {eventos['data_br'].min()} a {eventos['data_br'].max()}")
print(f"eventos depois do corte: {(eventos['data_br'] >= FIM_CORTE).sum()}")
assert (eventos["data_br"] < FIM_CORTE).all()

eventos: 275,565 linhas, de 2024-07-01 06:11:50 a 2026-09-30 22:59:33
eventos depois do corte: 0


In [6]:
base["ativa_no_corte"] = base["status_atual"] == "ativa"
assert (base["ativa_no_corte"] == base["data_fim"].isna()).all()

fim_efetivo = base["data_fim"].fillna(CORTE)
base["duracao_dias"] = (fim_efetivo - base["inicio"]).dt.days.astype("int64")
base["duracao_censurada"] = base["ativa_no_corte"]
assert (base["duracao_dias"] >= 0).all()

display(pd.crosstab(base["status_atual"], base["renovacao_automatica"], margins=True, margins_name="total"))
display(base.groupby(["plano_atual", "duracao_censurada"])["duracao_dias"].describe().round(1))

base = registrar("3. Situação no corte", "ativa_no_corte, duracao_dias, duracao_censurada", base)

renovacao_automatica,nao,sim,total
status_atual,,,
ativa,2250,12503,14753
expirada,17758,0,17758
reembolsada,72,0,72
total,20080,12503,32583


count   mean    std  min    25%    50%    75%     max
plano_atual duracao_censurada                                                         
anual       False               4549.0  597.6  400.3  5.0  365.0  365.0  730.0  2190.0
            True                8478.0  476.8  479.9  0.0  176.2  324.0  607.0  2213.0
mensal      False              13281.0  299.4  359.2  5.0   90.0  166.0  330.0  2160.0
            True                6275.0  338.4  427.4  0.0   71.0  202.0  425.0  2211.0

3. Situação no corte: 32,583 linhas (0 removidas)


## 4. Coorte anterior ao log

O log de eventos começa em 01/07/2024 e a `data_fim` mais antiga do arquivo é 07/07/2024: assinaturas que
começaram **e** terminaram antes de julho de 2024 não estão no arquivo. Das iniciadas antes de 01/07/2024,
só aparecem as que ainda existiam naquela data, e uma comparação por ano de início fica enviesada para quem ficou.
As linhas ficam, marcadas em `inicio_antes_do_log`, para que cada análise decida se as usa.

In [7]:
base["inicio_antes_do_log"] = base["inicio"] < INICIO_LOG

print(f"data_fim mais antiga: {base['data_fim'].min().date()}")
print(f"encerradas antes de {INICIO_LOG.date()}: {(base['data_fim'] < INICIO_LOG).sum()}")
assert (base["data_fim"] < INICIO_LOG).sum() == 0

display(pd.crosstab(base["inicio"].dt.year.rename("ano de início"),
                    base["ativa_no_corte"].map({True: "ativa no corte", False: "encerrada"}),
                    margins=True, margins_name="total"))

base = registrar("4. Coorte anterior ao log", "inicio_antes_do_log = inicio < 2024-07-01", base)

data_fim mais antiga: 2024-07-07
encerradas antes de 2024-07-01: 0


ativa_no_corte,ativa no corte,encerrada,total
ano de início,,,
2020,189,218,407
2021,468,898,1366
2022,319,989,1308
2023,332,1097,1429
2024,1361,4467,5828
2025,4525,7708,12233
2026,7559,2453,10012
total,14753,17830,32583


4. Coorte anterior ao log: 32,583 linhas (0 removidas)


## 5. Conferências

- **Número-prova:** 14.753 assinantes pagantes ativos no corte (notebook 01).
- 17.830 encerradas: as mesmas 17.830 `assinatura_expirada` do log.
- Iniciadas desde 01/07/2024: as mesmas contas e as mesmas datas dos `assinatura_iniciada` do log.
- Uma assinatura por conta: o dicionário diz que a pessoa pode sair e voltar, mas nem o retrato nem o log
  têm conta com mais de uma assinatura (pergunta em aberto com o dono do dado).

In [8]:
inicios_log = eventos.loc[eventos["evento"] == "assinatura_iniciada", ["user_id", "data_br"]]
expiradas_log = eventos.loc[eventos["evento"] == "assinatura_expirada", ["user_id", "data_br"]]
desde_log = base.loc[~base["inicio_antes_do_log"], ["user_id", "inicio"]]

datas_inicio = desde_log.merge(inicios_log, on="user_id", how="outer", validate="one_to_one", indicator=True)
datas_fim = (base.loc[base["data_fim"].notna(), ["user_id", "data_fim"]]
             .merge(expiradas_log, on="user_id", how="outer", validate="one_to_one", indicator=True))

conferencias = {
    "ativas no corte (número-prova 14.753)": (base["ativa_no_corte"].sum(), 14_753),
    "encerradas = assinatura_expirada do log": (base["data_fim"].notna().sum(), len(expiradas_log)),
    "iniciadas desde 01/07/2024 = assinatura_iniciada do log": (len(desde_log), len(inicios_log)),
    "inícios sem par no log (dos dois lados)": ((datas_inicio["_merge"] != "both").sum(), 0),
    "inícios com data diferente da do log": (
        (datas_inicio["inicio"] != datas_inicio["data_br"].dt.normalize()).sum(), 0),
    "fins sem par no log (dos dois lados)": ((datas_fim["_merge"] != "both").sum(), 0),
    "fins com data diferente da do log": (
        (datas_fim["data_fim"] != datas_fim["data_br"].dt.normalize()).sum(), 0),
    "contas com mais de uma assinatura": (base["user_id"].duplicated().sum(), 0),
    "contas com mais de um início no log": (inicios_log["user_id"].duplicated().sum(), 0),
}
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["bate"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["bate"].all()

,obtido,esperado,bate
ativas no corte (número-prova 14.753),14753,14753,True
encerradas = assinatura_expirada do log,17830,17830,True
iniciadas desde 01/07/2024 = assinatura_iniciada do log,27383,27383,True
inícios sem par no log (dos dois lados),0,0,True
inícios com data diferente da do log,0,0,True
fins sem par no log (dos dois lados),0,0,True
fins com data diferente da do log,0,0,True
contas com mais de uma assinatura,0,0,True
contas com mais de um início no log,0,0,True


## 6. Saídas e integridade dos brutos

In [9]:
COLUNAS = ["assinatura_id", "user_id", "plano_atual", "canal_cobranca", "inicio", "status_atual",
           "renovacao_automatica", "data_fim", "motivo_fim",
           "ativa_no_corte", "duracao_dias", "duracao_censurada", "inicio_antes_do_log"]
limpa = base[COLUNAS].reset_index(drop=True)

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / "assinaturas_limpa.parquet"
limpa.to_parquet(destino, index=False)

relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_assinaturas_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame())
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\assinaturas_limpa.parquet: 32,583 linhas x 13 colunas


,tipo
assinatura_id,object
user_id,object
plano_atual,object
canal_cobranca,object
inicio,datetime64[ns]
status_atual,object
renovacao_automatica,object
data_fim,datetime64[ns]
motivo_fim,object
ativa_no_corte,bool


,passo,regra,linhas,removidas
0,0. Arquivo bruto,assinaturas.csv sem filtro,32643,0
1,1. Sem contas de teste,e-mail não termina em @vitaliza.com.br,32583,60
2,2. Tipos convertidos,inicio e data_fim como data; motivo_fim vazio ...,32583,0
3,3. Situação no corte,"ativa_no_corte, duracao_dias, duracao_censurada",32583,0
4,4. Coorte anterior ao log,inicio_antes_do_log = inicio < 2024-07-01,32583,0


dados/brutos/ inalterado: True
